# Programa de fidelización de clientes

### Clasificación por rangos y comparación de descuentos

**ADMI 1301 · Introducción a la Programación**  
**Integrantes:** Mariagna Arias y Alejandro González  
**Datos:** ficticios, incluidos en este notebook

Este notebook permite explorar cómo una tienda puede reconocer a sus clientes, calcular descuentos y comparar sus efectos sobre la ganancia.


## 1. ¿Para quién es y qué problema resuelve?

**Para quién:** una tienda en línea pequeña que tiene un programa de fidelización y quiere recompensar a sus mejores clientes.

**Problema:** los descuentos se asignan "a ojo", lo que genera errores y tratos desiguales, y el negocio no sabe cuánto le cuestan realmente.

**Idea de negocio:** clasificar a cada cliente en un rango (Básico, Plata, Oro o Platino) según lo que ha gastado en el año, asignarle automáticamente el descuento de su rango y comparar escenarios de descuentos para decidir cuál conviene más, sin perder dinero en la venta.

**Qué hace la aplicación:**
1. Clasifica a cada cliente por rango.
2. Calcula su descuento (según el rango y su antigüedad) y lo limita para proteger un margen mínimo de ganancia.
3. Simula la compra de cada cliente respetando el inventario disponible.
4. Resume los resultados por rango.
5. Compara varios escenarios de descuentos y muestra su efecto en ingresos y ganancias.
6. Permite añadir datos nuevos: registrar clientes, compras repetidas de un mismo cliente y gastos en dinero, con actualización automática del rango y del inventario.

## 2. Cómo usar la aplicación

1. Abre este archivo en **Google Colab**, **Jupyter** o **VS Code** y elige **Ejecutar todo / Run all**. Las tablas aparecerán debajo de las celdas.
2. Explora las tablas de clientes, rangos y escenarios en las secciones 6 y 7.
3. En la sección 8, cambia `nombre_consulta` para consultar a otra persona.
4. En la sección 9, prueba el registro de clientes y compras. Para usar el menú por teclado, sigue las instrucciones de la sección 9.2.

**Para empezar de nuevo:** reinicia el entorno y ejecuta todas las celdas en orden. Los cambios y registros solo permanecen durante la sesión; ejecutar de nuevo una celda de registro repite la operación. Los datos son ficticios y no se necesitan archivos adicionales.


## 3. Uso de inteligencia artificial

Se utilizó **Claude (Anthropic)** para apoyar la estructura inicial del código, los comentarios y los datos de ejemplo; y **Codex (OpenAI)** para revisar y corregir el notebook.

Para verificar los resultados, se ejecutaron las celdas en orden y se revisaron y corrigieron partes del código, además de probar los límites de clasificación, los descuentos y el inventario.


## 4. Reglas de negocio y datos de prueba

Los rangos se definen por el **gasto acumulado del año** (en pesos colombianos). Cada rango tiene un descuento base, y los clientes con 3 o más años de antigüedad reciben un bono adicional.

In [1]:
# --- Reglas de los rangos: (nombre, gasto mínimo del año) ---
# La lista va de mayor a menor para clasificar con un ciclo.
RANGOS = [
    ("Platino", 3000000),
    ("Oro", 1500000),
    ("Plata", 500000),
    ("Básico", 0),
]

ANTIGUEDAD_MINIMA = 3      # años para recibir el bono
MARGEN_MINIMO = 0.10       # el precio final debe cubrir el costo + 10 %

# --- Productos de la tienda (precio y costo en pesos) ---
productos = [
    {"nombre": "Audífonos", "precio": 120000, "costo": 70000, "stock": 8},
    {"nombre": "Cargador",  "precio": 60000,  "costo": 30000, "stock": 7},
    {"nombre": "Termo",     "precio": 80000,  "costo": 45000, "stock": 8},
    {"nombre": "Mochila",   "precio": 150000, "costo": 90000, "stock": 6},
]

# --- Clientes ficticios: gasto del año, años de antigüedad y su compra ---
clientes = [
    {"nombre": "Laura Gómez",     "gasto_anual": 3800000, "antiguedad": 5, "producto": "Mochila",   "cantidad": 2},
    {"nombre": "Carlos Pérez",    "gasto_anual": 3100000, "antiguedad": 2, "producto": "Audífonos", "cantidad": 3},
    {"nombre": "Ana Torres",      "gasto_anual": 2200000, "antiguedad": 4, "producto": "Termo",     "cantidad": 4},
    {"nombre": "Juan Rodríguez",  "gasto_anual": 1700000, "antiguedad": 1, "producto": "Cargador",  "cantidad": 3},
    {"nombre": "Sofía Martínez",  "gasto_anual": 1550000, "antiguedad": 3, "producto": "Audífonos", "cantidad": 2},
    {"nombre": "Diego Castro",    "gasto_anual": 900000,  "antiguedad": 2, "producto": "Mochila",   "cantidad": 1},
    {"nombre": "Valentina Ruiz",  "gasto_anual": 650000,  "antiguedad": 4, "producto": "Termo",     "cantidad": 2},
    {"nombre": "Andrés Molina",   "gasto_anual": 520000,  "antiguedad": 1, "producto": "Cargador",  "cantidad": 5},
    {"nombre": "Camila Vargas",   "gasto_anual": 300000,  "antiguedad": 3, "producto": "Audífonos", "cantidad": 1},
    {"nombre": "Felipe Ortiz",    "gasto_anual": 180000,  "antiguedad": 1, "producto": "Termo",     "cantidad": 2},
    {"nombre": "Daniela Herrera", "gasto_anual": 90000,   "antiguedad": 0, "producto": "Cargador",  "cantidad": 1},
    {"nombre": "Mateo Silva",     "gasto_anual": 2900000, "antiguedad": 6, "producto": "Audífonos", "cantidad": 4},
]

print("Clientes:", len(clientes), "| Productos:", len(productos))

Clientes: 12 | Productos: 4


## 5. Funciones de la aplicación

Cada función hace una sola tarea, así es más fácil probarla y explicarla.

In [2]:
def formato_cop(valor):
    """Muestra un número como pesos: 1500000 -> $1.500.000"""
    return "$" + f"{valor:,.0f}".replace(",", ".")


def clasificar_rango(gasto_anual):
    """Devuelve el rango del cliente según su gasto del año."""
    if not isinstance(gasto_anual, (int, float)) or isinstance(gasto_anual, bool) or gasto_anual < 0:
        raise ValueError("El gasto anual debe ser un número no negativo.")
    for nombre, minimo in RANGOS:
        if gasto_anual >= minimo:
            return nombre
    return "Básico"


def calcular_descuento(rango, antiguedad, escenario):
    """Descuento base del rango + bono por antigüedad."""
    descuento = escenario["descuentos"][rango]
    if antiguedad >= ANTIGUEDAD_MINIMA:
        descuento = descuento + escenario["bono_antiguedad"]
    if not 0 <= descuento <= 1:
        raise ValueError("El descuento total debe estar entre 0 y 100 %.")
    return descuento


def limitar_por_margen(descuento, precio, costo):
    """Evita que el descuento deje el precio por debajo de costo + margen mínimo."""
    if not 0 <= descuento <= 1:
        raise ValueError("El descuento debe estar entre 0 y 100 %.")
    if precio <= 0 or costo < 0:
        raise ValueError("El precio debe ser positivo y el costo no negativo.")
    if precio < costo * (1 + MARGEN_MINIMO):
        raise ValueError("El precio no cubre el costo más el margen mínimo.")
    descuento_maximo = 1 - (costo * (1 + MARGEN_MINIMO)) / precio
    if descuento_maximo < 0:
        descuento_maximo = 0
    if descuento > descuento_maximo:
        return descuento_maximo
    return descuento


def buscar_producto(nombre, lista_productos):
    """Busca un producto por nombre dentro de la lista."""
    for producto in lista_productos:
        if producto["nombre"] == nombre:
            return producto
    return None


def simular_compra(cliente, stock, escenario):
    """
    Simula la compra de un cliente en un escenario.
    Respeta el inventario: si no alcanza el stock, vende lo disponible
    y el resto queda pendiente.
    """
    producto = buscar_producto(cliente["producto"], productos)
    if producto is None:
        raise ValueError("Producto desconocido: " + str(cliente["producto"]))
    rango = clasificar_rango(cliente["gasto_anual"])

    # Inventario
    pedidas = cliente["cantidad"]
    if not isinstance(pedidas, int) or isinstance(pedidas, bool) or pedidas <= 0:
        raise ValueError("La cantidad debe ser un entero positivo.")
    if pedidas <= stock[producto["nombre"]]:
        vendidas = pedidas
    else:
        vendidas = stock[producto["nombre"]]
    pendientes = pedidas - vendidas
    stock[producto["nombre"]] = stock[producto["nombre"]] - vendidas

    # Descuento (con tope de margen)
    descuento = calcular_descuento(rango, cliente["antiguedad"], escenario)
    descuento = limitar_por_margen(descuento, producto["precio"], producto["costo"])

    # Dinero
    precio_final = producto["precio"] * (1 - descuento)
    ingreso = precio_final * vendidas
    costo_total = producto["costo"] * vendidas
    descuento_otorgado = producto["precio"] * descuento * vendidas

    return {
        "cliente": cliente["nombre"],
        "rango": rango,
        "descuento": descuento,
        "vendidas": vendidas,
        "pendientes": pendientes,
        "ingreso": ingreso,
        "ganancia": ingreso - costo_total,
        "descuento_otorgado": descuento_otorgado,
    }


def ejecutar_escenario(escenario):
    """Procesa a todos los clientes con un escenario y devuelve la lista de resultados."""
    # Cada escenario parte del mismo inventario inicial
    stock = {}
    for producto in productos:
        stock[producto["nombre"]] = producto["stock"]

    resultados = []
    for cliente in clientes:
        # Los clientes nuevos que aún no han comprado no entran a la simulación
        if cliente["producto"] is None:
            continue
        resultados.append(simular_compra(cliente, stock, escenario))
    return resultados


def imprimir_tabla(encabezados, filas):
    """Muestra una tabla HTML en un notebook o texto alineado fuera de él."""
    try:
        from IPython.display import display, HTML
        from html import escape
        cabecera = "".join("<th style='text-align:left;padding:10px 14px;background:#eaf1f8;border-bottom:2px solid #b7c9dc'>" + escape(str(h)) + "</th>" for h in encabezados)
        cuerpo = ""
        for i, fila in enumerate(filas):
            color = "#f6f9fc" if i % 2 == 0 else "#ffffff"
            celdas = "".join("<td style='padding:9px 14px;border-bottom:1px solid #e1e8ef'>" + escape(str(v)) + "</td>" for v in fila)
            cuerpo += "<tr style='background:" + color + "'>" + celdas + "</tr>"
        display(HTML("<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:14px;width:auto'><thead><tr>" + cabecera + "</tr></thead><tbody>" + cuerpo + "</tbody></table></div>"))
    except ImportError:
        anchos = [max([len(str(encabezados[i]))] + [len(str(f[i])) for f in filas]) for i in range(len(encabezados))]
        print("  ".join(str(h).ljust(anchos[i]) for i, h in enumerate(encabezados)))
        print("  ".join("─" * ancho for ancho in anchos))
        for fila in filas:
            print("  ".join(str(v).ljust(anchos[i]) for i, v in enumerate(fila)))


## 6. Escenarios de descuentos

Un **escenario** define el descuento de cada rango y el bono por antigüedad. Aquí se definen cuatro, y el primero (sin descuentos) sirve como punto de comparación.

In [3]:
escenarios = [
    {
        "nombre": "Sin descuentos",
        "descuentos": {"Básico": 0.00, "Plata": 0.00, "Oro": 0.00, "Platino": 0.00},
        "bono_antiguedad": 0.00,
    },
    {
        "nombre": "Conservador",
        "descuentos": {"Básico": 0.00, "Plata": 0.03, "Oro": 0.06, "Platino": 0.09},
        "bono_antiguedad": 0.01,
    },
    {
        "nombre": "Base",
        "descuentos": {"Básico": 0.00, "Plata": 0.05, "Oro": 0.10, "Platino": 0.15},
        "bono_antiguedad": 0.02,
    },
    {
        "nombre": "Agresivo",
        "descuentos": {"Básico": 0.05, "Plata": 0.10, "Oro": 0.20, "Platino": 0.30},
        "bono_antiguedad": 0.03,
    },
]

# Escenario que usaremos para el detalle (índice 2 = "Base")
escenario_base = escenarios[2]
print("Escenario seleccionado:", escenario_base["nombre"])

Escenario seleccionado: Base


### 6.1 Resultado por cliente (escenario Base)

**Tabla 1.** Clasificación, descuento y compra de cada cliente. Muestra para quién se aplica cada descuento y qué pedidos quedan pendientes por falta de inventario.

In [4]:
resultados_base = ejecutar_escenario(escenario_base)

filas = []
for r in resultados_base:
    filas.append([
        r["cliente"],
        r["rango"],
        f"{r['descuento'] * 100:.1f} %",
        r["vendidas"],
        r["pendientes"],
        formato_cop(r["ingreso"]),
        formato_cop(r["ganancia"]),
    ])

print("Tabla 1. Clientes en el escenario Base")
imprimir_tabla(
    ["Cliente", "Rango", "Descuento", "Vendidas", "Pendientes", "Ingreso", "Ganancia"],
    filas,
)

Tabla 1. Clientes en el escenario Base


Cliente,Rango,Descuento,Vendidas,Pendientes,Ingreso,Ganancia
Laura Gómez,Platino,17.0 %,2,0,$249.000,$69.000
Carlos Pérez,Platino,15.0 %,3,0,$306.000,$96.000
Ana Torres,Oro,12.0 %,4,0,$281.600,$101.600
Juan Rodríguez,Oro,10.0 %,3,0,$162.000,$72.000
Sofía Martínez,Oro,12.0 %,2,0,$211.200,$71.200
Diego Castro,Plata,5.0 %,1,0,$142.500,$52.500
Valentina Ruiz,Plata,7.0 %,2,0,$148.800,$58.800
Andrés Molina,Plata,5.0 %,4,1,$228.000,$108.000
Camila Vargas,Básico,2.0 %,1,0,$117.600,$47.600
Felipe Ortiz,Básico,0.0 %,2,0,$160.000,$70.000


### 6.2 Resumen por rango (escenario Base)

**Tabla 2.** Cuántos clientes hay en cada rango y cuánto se les descuenta en total.

In [5]:
nombres_rangos = ["Platino", "Oro", "Plata", "Básico"]

filas = []
for rango in nombres_rangos:
    cantidad = 0
    ingresos = 0
    ganancias = 0
    descuentos_total = 0
    for r in resultados_base:
        if r["rango"] == rango:
            cantidad = cantidad + 1
            ingresos = ingresos + r["ingreso"]
            ganancias = ganancias + r["ganancia"]
            descuentos_total = descuentos_total + r["descuento_otorgado"]
    filas.append([
        rango,
        cantidad,
        formato_cop(ingresos),
        formato_cop(descuentos_total),
        formato_cop(ganancias),
    ])

print("Tabla 2. Resumen por rango (escenario Base)")
imprimir_tabla(["Rango", "Clientes", "Ingresos", "Descuentos otorgados", "Ganancia"], filas)

Tabla 2. Resumen por rango (escenario Base)


Rango,Clientes,Ingresos,Descuentos otorgados,Ganancia
Platino,2,$555.000,$105.000,$165.000
Oro,4,$866.000,$114.000,$316.000
Plata,3,$519.300,$30.700,$219.300
Básico,3,$277.600,$2.400,$117.600


## 7. Comparación de escenarios

**Tabla 3.** Efecto de cada escenario en ingresos y ganancias.

La última columna responde: *¿cuánto tendrían que aumentar las ventas para que la ganancia fuera igual a la de "Sin descuentos"?* Se calcula como `ganancia sin descuentos / ganancia del escenario − 1`, y supone que la ganancia crece en proporción a las ventas.

In [6]:
# Primero calculamos la ganancia sin descuentos (punto de comparación)
resultados_sin = ejecutar_escenario(escenarios[0])
ganancia_sin = 0
ingreso_sin = 0
for r in resultados_sin:
    ganancia_sin = ganancia_sin + r["ganancia"]
    ingreso_sin = ingreso_sin + r["ingreso"]

filas = []
for escenario in escenarios:
    resultados = ejecutar_escenario(escenario)
    ingreso = 0
    ganancia = 0
    descuentos_total = 0
    for r in resultados:
        ingreso = ingreso + r["ingreso"]
        ganancia = ganancia + r["ganancia"]
        descuentos_total = descuentos_total + r["descuento_otorgado"]

    if ganancia_sin != 0:
        texto_variacion = f"{(ganancia - ganancia_sin) / ganancia_sin * 100:+.1f} %"
    else:
        texto_variacion = "No calculable"
    if ganancia <= 0 and ganancia_sin > 0:
        texto_aumento = "No calculable"
    elif ganancia < ganancia_sin:
        aumento_necesario = (ganancia_sin / ganancia - 1) * 100
        texto_aumento = f"{aumento_necesario:.1f} %"
    else:
        texto_aumento = "0.0 %"

    filas.append([
        escenario["nombre"],
        formato_cop(ingreso),
        formato_cop(descuentos_total),
        formato_cop(ganancia),
        texto_variacion,
        texto_aumento,
    ])

print("Tabla 3. Comparación de escenarios")
imprimir_tabla(
    ["Escenario", "Ingresos", "Descuentos", "Ganancia", "Cambio ganancia", "Ventas extra necesarias"],
    filas,
)

Tabla 3. Comparación de escenarios


Escenario,Ingresos,Descuentos,Ganancia,Cambio ganancia,Ventas extra necesarias
Sin descuentos,$2.470.000,$0,$1.070.000,+0.0 %,0.0 %
Conservador,$2.321.500,$148.500,$921.500,-13.9 %,16.1 %
Base,$2.217.900,$252.100,$817.900,-23.6 %,30.8 %
Agresivo,$1.965.600,$504.400,$565.600,-47.1 %,89.2 %


## 8. Consultar un cliente

Cambien el nombre de la variable para ver el rango de un cliente, su descuento y cuánto le falta para subir de rango.

In [7]:
nombre_consulta = "Sofía Martínez"   # <- cambien este nombre por otro cliente de la lista

encontrado = None
for cliente in clientes:
    if cliente["nombre"] == nombre_consulta:
        encontrado = cliente

if encontrado is None:
    print("No encontramos a", nombre_consulta)
else:
    rango = clasificar_rango(encontrado["gasto_anual"])
    descuento = calcular_descuento(rango, encontrado["antiguedad"], escenario_base)
    print("Cliente:", encontrado["nombre"])
    print("Gasto del año:", formato_cop(encontrado["gasto_anual"]))
    print("Rango:", rango)
    print(f"Descuento en el escenario Base: {descuento * 100:.1f} %")

    # ¿Cuánto falta para el siguiente rango?
    siguiente = None
    for nombre, minimo in RANGOS:
        if minimo > encontrado["gasto_anual"]:
            siguiente = (nombre, minimo)   # el último que cumpla es el más cercano
    if siguiente is None:
        print("Ya está en el rango más alto.")
    else:
        falta = siguiente[1] - encontrado["gasto_anual"]
        print("Le faltan", formato_cop(falta), "para llegar a", siguiente[0])

Cliente: Sofía Martínez
Gasto del año: $1.550.000
Rango: Oro
Descuento en el escenario Base: 12.0 %
Le faltan $1.450.000 para llegar a Platino


## 9. Registrar clientes, compras y gastos

Hasta aquí trabajamos con los datos iniciales. Esta sección permite **añadir datos nuevos**:
- **Registrar un cliente nuevo.**
- **Registrar una compra** de un cliente (puede repetirse: un mismo cliente puede comprar varias veces). El valor de la compra, ya con descuento, se suma a su gasto del año y el inventario se descuenta.
- **Registrar un gasto** de cierta cantidad de dinero (por ejemplo, una compra hecha en otro canal).

Cada vez que se suma dinero, el programa vuelve a calcular el rango y avisa si el cliente **sube de rango**. El descuento de una compra se calcula con el rango que el cliente tenía **antes** de comprar.

Nota: cada celda que registre datos agrega información cada vez que se ejecuta. Si repiten una celda, el dato se registrará dos veces.
Las comparaciones de escenarios usan únicamente los pedidos de ejemplo originales y el inventario disponible al momento de ejecutarlas; no reconstruyen el historial de compras. Reinicien y ejecuten todo para recuperar el estado inicial. Los datos se guardan solo durante la sesión.


In [8]:
historial_compras = []              # aquí se guarda cada compra o gasto registrado
escenario_actual = escenarios[2]    # escenario de descuentos para las compras nuevas (Base)


def buscar_cliente(nombre):
    """Busca un cliente por nombre. Devuelve None si no existe."""
    for cliente in clientes:
        if cliente["nombre"] == nombre:
            return cliente
    return None


def sumar_gasto(cliente, monto):
    """Suma dinero al gasto del año y avisa si el cliente cambia de rango."""
    rango_antes = clasificar_rango(cliente["gasto_anual"])
    cliente["gasto_anual"] = cliente["gasto_anual"] + monto
    rango_despues = clasificar_rango(cliente["gasto_anual"])
    if rango_antes != rango_despues:
        print(f"¡{cliente['nombre']} subió de {rango_antes} a {rango_despues}!")


def registrar_cliente(nombre, antiguedad=0, gasto_inicial=0):
    """Agrega un cliente nuevo a la lista."""
    if nombre.strip() == "":
        print("El nombre no puede estar vacío.")
        return
    if buscar_cliente(nombre) is not None:
        print("Ya existe un cliente con el nombre", nombre)
        return
    if not isinstance(antiguedad, int) or isinstance(antiguedad, bool) or antiguedad < 0 or not isinstance(gasto_inicial, (int, float)) or isinstance(gasto_inicial, bool) or gasto_inicial < 0:
        print("La antigüedad debe ser entera y el gasto numérico; ambos deben ser no negativos.")
        return
    clientes.append({
        "nombre": nombre,
        "gasto_anual": gasto_inicial,
        "antiguedad": antiguedad,
        "producto": None,    # todavía no tiene una compra registrada
        "cantidad": 0,
    })
    print(f"Cliente registrado: {nombre} (rango {clasificar_rango(gasto_inicial)})")


def registrar_compra(nombre, nombre_producto, cantidad):
    """Registra una compra: aplica el descuento, baja el inventario y suma al gasto."""
    cliente = buscar_cliente(nombre)
    producto = buscar_producto(nombre_producto, productos)
    if cliente is None:
        print("No existe el cliente", nombre)
        return
    if producto is None:
        print("No existe el producto", nombre_producto)
        return
    if not isinstance(cantidad, int) or isinstance(cantidad, bool) or cantidad <= 0:
        print("La cantidad debe ser un número entero mayor que cero.")
        return

    # Inventario
    if cantidad <= producto["stock"]:
        vendidas = cantidad
    else:
        vendidas = producto["stock"]
    pendientes = cantidad - vendidas
    if vendidas == 0:
        print("No hay inventario de", nombre_producto)
        return

    # Descuento según el rango ANTES de esta compra
    rango = clasificar_rango(cliente["gasto_anual"])
    descuento = calcular_descuento(rango, cliente["antiguedad"], escenario_actual)
    descuento = limitar_por_margen(descuento, producto["precio"], producto["costo"])
    total = producto["precio"] * (1 - descuento) * vendidas

    # Actualizar los datos
    producto["stock"] = producto["stock"] - vendidas
    historial_compras.append({
        "cliente": nombre,
        "detalle": f"{vendidas} x {nombre_producto}",
        "rango": rango,
        "descuento": descuento,
        "total": total,
    })

    print(f"Compra registrada: {nombre} ({rango}) compró {vendidas} x {nombre_producto} "
          f"con {descuento * 100:.1f} % de descuento por {formato_cop(total)}.")
    if pendientes > 0:
        print(f"Atención: quedaron {pendientes} unidades pendientes por falta de inventario.")
    sumar_gasto(cliente, total)


def registrar_gasto(nombre, monto):
    """Suma un gasto en dinero (por ejemplo, una compra en otro canal)."""
    cliente = buscar_cliente(nombre)
    if cliente is None:
        print("No existe el cliente", nombre)
        return
    if monto <= 0:
        print("El monto debe ser mayor que cero.")
        return
    historial_compras.append({
        "cliente": nombre,
        "detalle": "Gasto en dinero",
        "rango": clasificar_rango(cliente["gasto_anual"]),
        "descuento": 0,
        "total": monto,
    })
    print(f"Gasto registrado: {nombre} gastó {formato_cop(monto)}.")
    sumar_gasto(cliente, monto)


def mostrar_clientes():
    """Tabla con el estado actual de todos los clientes."""
    filas = []
    for cliente in clientes:
        rango = clasificar_rango(cliente["gasto_anual"])
        descuento = calcular_descuento(rango, cliente["antiguedad"], escenario_actual)
        filas.append([
            cliente["nombre"],
            rango,
            formato_cop(cliente["gasto_anual"]),
            cliente["antiguedad"],
            f"{descuento * 100:.1f} %",
        ])
    imprimir_tabla(["Cliente", "Rango", "Gasto del año", "Antigüedad", "Descuento"], filas)


def mostrar_historial():
    """Tabla con las compras y gastos registrados en esta sesión."""
    filas = []
    for registro in historial_compras:
        filas.append([
            registro["cliente"],
            registro["detalle"],
            registro["rango"],
            f"{registro['descuento'] * 100:.1f} %",
            formato_cop(registro["total"]),
        ])
    imprimir_tabla(["Cliente", "Detalle", "Rango al comprar", "Descuento", "Total"], filas)

### 9.1 Ejemplo: registrar datos nuevos

Aquí se registra un cliente nuevo, varias compras del mismo cliente y un gasto adicional. Observen cómo el rango cambia a medida que se acumula el gasto.

In [9]:
# 1. Un cliente nuevo
registrar_cliente("Paula Ríos", antiguedad=2)

# 2. El mismo cliente hace varias compras: su gasto se va acumulando
registrar_compra("Paula Ríos", "Mochila", 2)
registrar_compra("Paula Ríos", "Audífonos", 3)

# 3. Un cliente que ya existía compra más y sube de rango
registrar_compra("Camila Vargas", "Audífonos", 3)

# 4. Un gasto en dinero hecho en otro canal
registrar_gasto("Felipe Ortiz", 500000)

Cliente registrado: Paula Ríos (rango Básico)
Compra registrada: Paula Ríos (Básico) compró 2 x Mochila con 0.0 % de descuento por $300.000.
Compra registrada: Paula Ríos (Básico) compró 3 x Audífonos con 0.0 % de descuento por $360.000.
¡Paula Ríos subió de Básico a Plata!
Compra registrada: Camila Vargas (Básico) compró 3 x Audífonos con 2.0 % de descuento por $352.800.
¡Camila Vargas subió de Básico a Plata!
Gasto registrado: Felipe Ortiz gastó $500.000.
¡Felipe Ortiz subió de Básico a Plata!


**Tabla 4.** Compras y gastos registrados.

In [10]:
print("Tabla 4. Historial de compras y gastos registrados")
mostrar_historial()

Tabla 4. Historial de compras y gastos registrados


Cliente,Detalle,Rango al comprar,Descuento,Total
Paula Ríos,2 x Mochila,Básico,0.0 %,$300.000
Paula Ríos,3 x Audífonos,Básico,0.0 %,$360.000
Camila Vargas,3 x Audífonos,Básico,2.0 %,$352.800
Felipe Ortiz,Gasto en dinero,Básico,0.0 %,$500.000


**Tabla 5.** Estado actual de los clientes después de registrar los datos nuevos (el gasto y el rango ya están actualizados).

In [11]:
print("Tabla 5. Estado actual de los clientes")
mostrar_clientes()

Tabla 5. Estado actual de los clientes


Cliente,Rango,Gasto del año,Antigüedad,Descuento
Laura Gómez,Platino,$3.800.000,5,17.0 %
Carlos Pérez,Platino,$3.100.000,2,15.0 %
Ana Torres,Oro,$2.200.000,4,12.0 %
Juan Rodríguez,Oro,$1.700.000,1,10.0 %
Sofía Martínez,Oro,$1.550.000,3,12.0 %
Diego Castro,Plata,$900.000,2,5.0 %
Valentina Ruiz,Plata,$650.000,4,7.0 %
Andrés Molina,Plata,$520.000,1,5.0 %
Camila Vargas,Plata,$652.800,3,7.0 %
Felipe Ortiz,Plata,$680.000,1,5.0 %


### 9.2 Menú interactivo (opcional)

Para registrar datos escribiéndolos con el teclado, quiten el `#` de la última línea y ejecuten la celda. Mientras el menú esté abierto, el notebook espera que escriban; elijan **6** para salir. (Está comentado para que **Run All** no se quede esperando.)

In [12]:
def leer_numero(mensaje):
    """Pide un número entero positivo o cero y repite hasta que sea válido."""
    texto = input(mensaje).strip()
    while not texto.isdigit():
        print("Por favor escriban solo números, sin puntos ni signos.")
        texto = input(mensaje).strip()
    return int(texto)


def menu_interactivo():
    """Menú para registrar clientes, compras y gastos desde el teclado."""
    opcion = ""
    while opcion != "6":
        print()
        print("=== MENÚ ===")
        print("1. Registrar cliente nuevo")
        print("2. Registrar compra")
        print("3. Registrar gasto en dinero")
        print("4. Ver clientes")
        print("5. Ver historial")
        print("6. Salir")
        opcion = input("Elijan una opción: ").strip()

        if opcion == "1":
            nombre = input("Nombre del cliente: ").strip()
            antiguedad = leer_numero("Años de antigüedad: ")
            registrar_cliente(nombre, antiguedad)
        elif opcion == "2":
            nombre = input("Nombre del cliente: ").strip()
            print("Productos:", end=" ")
            for producto in productos:
                print(f"{producto['nombre']} (stock {producto['stock']})", end="  ")
            print()
            nombre_producto = input("Producto: ").strip()
            cantidad = leer_numero("Cantidad: ")
            registrar_compra(nombre, nombre_producto, cantidad)
        elif opcion == "3":
            nombre = input("Nombre del cliente: ").strip()
            monto = leer_numero("Monto gastado en pesos: ")
            registrar_gasto(nombre, monto)
        elif opcion == "4":
            mostrar_clientes()
        elif opcion == "5":
            mostrar_historial()
        elif opcion != "6":
            print("Opción no válida.")
    print("Hasta pronto.")


# menu_interactivo()    # <- quiten el # para usar el menú

## 10. Pruebas de funcionamiento

Las siguientes pruebas comprueban los límites de los rangos, el cálculo de descuentos, el margen mínimo y el inventario. Utilizan datos de prueba independientes y no alteran los clientes ni productos de la aplicación. Ejecuten la celda para repetirlas.


In [13]:
# Pruebas de los límites: el valor exacto entra al rango superior.
casos = [(0, "Básico"), (499999, "Básico"), (500000, "Plata"),
         (1499999, "Plata"), (1500000, "Oro"),
         (2999999, "Oro"), (3000000, "Platino")]
for gasto, esperado in casos:
    assert clasificar_rango(gasto) == esperado

# Bono de antigüedad y tope para proteger el costo más 10 %.
assert abs(calcular_descuento("Oro", 3, escenario_base) - 0.12) < 0.000001
assert abs(limitar_por_margen(0.50, 100, 80) - 0.12) < 0.000001
try:
    limitar_por_margen(0.10, 100, 95)
    raise AssertionError("Se aceptó un precio que no cubre el margen mínimo")
except ValueError:
    pass

# Pedido superior al inventario: una unidad vendida y dos pendientes.
stock_prueba = {"Mochila": 1}
cliente_prueba = {"nombre": "Prueba", "gasto_anual": 0, "antiguedad": 0,
                  "producto": "Mochila", "cantidad": 3}
resultado_prueba = simular_compra(cliente_prueba, stock_prueba, escenarios[0])
assert resultado_prueba["vendidas"] == 1
assert resultado_prueba["pendientes"] == 2
assert stock_prueba["Mochila"] == 0
assert abs(resultado_prueba["ingreso"] - 150000) < 0.01
assert abs(resultado_prueba["ganancia"] - 60000) < 0.01

# Todos los escenarios se calculan con el mismo inventario de partida.
primera = ejecutar_escenario(escenario_base)
segunda = ejecutar_escenario(escenario_base)
assert primera == segunda
print("Pruebas aprobadas: rangos, descuentos, margen, inventario y reproducibilidad.")


Pruebas aprobadas: rangos, descuentos, margen, inventario y reproducibilidad.


## 11. Conclusiones y limitaciones

Con los pedidos y el inventario ficticios, el escenario sin descuentos obtiene el mayor ingreso y ganancia: los demás escenarios reducen el precio sin suponer un aumento en las unidades vendidas. La tabla 3 muestra cuánto deberían aumentar las ventas para recuperar la ganancia inicial, bajo el supuesto de proporcionalidad. Estos resultados ayudan a comparar políticas, pero no predicen la reacción de los clientes.

Los descuentos no demuestran que aumenten las ventas ni prueban cuál estrategia maximiza beneficios reales. La comparación supone que la mezcla de productos, los costos unitarios y el inventario permiten aumentar las ventas proporcionalmente; el inventario de ejemplo podría impedirlo.

**Limitaciones:** datos ficticios de 12 clientes y 4 productos; la simulación inicial contiene un pedido por cliente; no se incluyen impuestos, envíos ni otros costos. El gasto del año determina el rango. Los registros nuevos se conservan solamente durante la sesión y no reconstruyen los escenarios históricos. El margen mínimo se calcula sobre el costo, no sobre el precio de venta.
